# Model selection & correlation statistics, worked on one GRB (GRB 110721A)

**Why this notebook exists.** The goal is not the astrophysics of one burst — it is to
*understand the statistics we quote*: **AIC / BIC / DIC**, **error propagation**
(and why parameter **covariance** matters), and the **Spearman correlation coefficient
and its p-value**. We use a single real GRB (110721A) and its Band vs Band+Blackbody
spectral fit as the concrete vehicle, then unpack each concept with runnable code and notes.

**How to run (standalone, outside any AI terminal).**
1. Use the `threeML` conda environment as the kernel (`conda activate threeML`, then
   select that Python 3 kernel in Jupyter).
2. Just *Run All*. The first cell sets the Fermi/CALDB environment variables itself
   (from `sys.prefix`) **before** importing `threeML`, so you do not need to export
   anything by hand. If your data live elsewhere, edit `ROOT` in the config cell.

**What it does, section by section**
- §1  Load GRB 110721A, background-fit, and show its **Bayesian-block** time segmentation.
- §2  Fit one Bayesian block with **Band** and **Band+Blackbody** (3ML / MLE).
- §3  Compare the two models with **AIC, BIC, DIC** — definitions, formulas, meaning.
- §4  **Error propagation**: the delta method, the `err/(v·ln10)` dex rule, and why the
      **Ep–kT covariance** changes the error on any derived quantity.
- §5  **Correlation**: Pearson vs **Spearman**, what its **p-value** is and assumes,
      and two failure modes (attenuation, and *spurious* correlation from covariant errors).
- §6  Synthesis: what you may and may not claim from an Ep–kT correlation.

## Setup

The **only** environment-specific requirement is that this notebook runs on the
`threeML` env's Python. The cell below makes `threeML` importable by exporting the
Fermitools/CALDB paths *before* the import (an unset `CALDBALIAS` makes `import threeML`
abort). It uses `os.environ.setdefault`, so if you already exported them, yours win.

In [ ]:
import os, sys
# --- make threeML importable standalone: CALDB must be set BEFORE importing threeML ---
_P = sys.prefix                                   # the active conda env (should be threeML)
_fermi = os.environ.setdefault("FERMI_DIR", f"{_P}/share/fermitools")
_caldb = f"{_fermi}/data/caldb"
os.environ.setdefault("CALDB",       _caldb)
os.environ.setdefault("CALDBCONFIG", f"{_caldb}/software/tools/caldb.config")
os.environ.setdefault("CALDBALIAS",  f"{_caldb}/software/tools/alias_config.fits")
os.environ.setdefault("CALDBROOT",   _caldb)
os.environ.setdefault("EXTFILESSYS", f"{_fermi}/refdata/fermi")
for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(v, "1")

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from astropy.io import fits
np.random.seed(0)

import threeML, astromodels
from threeML import (TimeSeriesBuilder, DataList, JointLikelihood, BayesianAnalysis,
                     Model, PointSource)
from astromodels import Band, Blackbody, Log_uniform_prior, Uniform_prior
from threeML.utils.statistics.stats_tools import Significance  # noqa (kept for reference)
print("threeML", threeML.__version__, "| astromodels", astromodels.__version__)

# --------------------------------------------------------------- config
# Edit ROOT if the repository / data live elsewhere.
ROOT = os.environ.get("SINGLEPULSES_ROOT", "/usr/local/disk1/SinglePulses")
GRB  = "bn110721200"
DATA = os.path.join(ROOT, "data", GRB)

# Approved detectors + windows for this burst (from the pipeline's Stage-1 catalog,
# results/background_intervals.ecsv). Hard-coded here so the notebook is self-contained.
NAI, BGO = "n9", "b1"                 # brightest NaI (theta=29.6 deg) + its BGO
BKG_PRE, BKG_POST = (-27.97, -4.97), (26.03, 75.03)   # quiet pre/post background
SRC = (-1.15, 18.93)                  # full emission window (union of the blocks)
NAI_RANGES = ("8.1-33", "40-900")     # 33-40 keV cut = NaI iodine K-edge (Gruber+2014)
BGO_RANGE  = ("300-40000",)

tte = lambda d: os.path.join(DATA, f"glg_tte_{d}_{GRB}_v00.fit.gz")
rsp = lambda d: os.path.join(DATA, f"glg_cspec_{d}_{GRB}_v03.rsp")
assert os.path.exists(tte(NAI)), f"missing data: {tte(NAI)} -- set ROOT to your repo"
print("data dir:", DATA)

## §1 · The data and its Bayesian-block segmentation

A GRB light curve is a stream of photon arrival times. **Bayesian Blocks**
(Scargle et al. 2013) partitions that stream into the *fewest* piecewise-constant-rate
segments justified by the data, controlled by a false-alarm prior `p0` (smaller `p0` →
fewer, more conservative blocks). Each block is a stretch over which the count rate is
statistically constant — a natural, data-driven time bin to extract a spectrum from.

Below we read the NaI (n9) TTE directly with `astropy` (transparent — no black box),
build the light curve, fit an off-source **polynomial background**, and overlay the
Bayesian blocks that `threeML` finds. We then pick the **brightest block** as the
interval we will fit spectrally in §2.

In [ ]:
# ---- raw light curve straight from the TTE (transparent) ----
with fits.open(tte(NAI)) as h:
    ev = h["EVENTS"].data["TIME"]
    # trigger time (MET); GBM stores it as TRIGTIME in a header
    trig = None
    for hdu in h:
        if "TRIGTIME" in hdu.header:
            trig = float(hdu.header["TRIGTIME"]); break
t = ev - trig                                    # seconds since trigger
dt = 0.064
edges = np.arange(-5, 30 + dt, dt)
cnt, _ = np.histogram(t, bins=edges)
ctr = 0.5 * (edges[:-1] + edges[1:])
rate = cnt / dt

# ---- Bayesian blocks via threeML (same engine the pipeline uses) ----
ts = TimeSeriesBuilder.from_gbm_tte(NAI, tte_file=tte(NAI), rsp_file=rsp(NAI))
ts.set_background_interval(f"{BKG_PRE[0]}-{BKG_PRE[1]}", f"{BKG_POST[0]}-{BKG_POST[1]}")
ts.create_time_bins(-2.0, 20.0, method="bayesblocks", p0=0.05)
bstart, bstop = np.asarray(ts.bins.starts), np.asarray(ts.bins.stops)
print(f"{len(bstart)} Bayesian blocks over [-2, 20] s (p0=0.05)")

# fit target = the block at the peak count rate (the νFν peak epoch, where a thermal
# component — if present — is expected to be strongest; cf. Axelsson 2012 for this burst).
peak_t = ctr[np.argmax(np.where((ctr > -2) & (ctr < 20), rate, -np.inf))]
kpk = int(np.where((bstart <= peak_t) & (bstop > peak_t))[0][0])
PK = (float(bstart[kpk]), float(bstop[kpk]))
blk_cnt = int(cnt[(ctr >= PK[0]) & (ctr < PK[1])].sum())
print(f"peak rate at t={peak_t:.2f}s → block #{kpk} = [{PK[0]:.3f}, {PK[1]:.3f}] s "
      f"({PK[1]-PK[0]:.2f}s, {blk_cnt} counts) — this is the block we fit")

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.step(ctr, rate, where="mid", lw=0.8, color="0.35")
for s in bstart: ax.axvline(s, color="#b3202c", lw=0.6, alpha=0.6)
ax.axvline(bstop[-1], color="#b3202c", lw=0.6, alpha=0.6)
ax.axvspan(PK[0], PK[1], color="#ffd54f", alpha=0.4, label=f"block #{kpk} (fit target)")
ax.set_xlim(-5, 25); ax.set_xlabel("Time since trigger (s)"); ax.set_ylabel("counts / s (n9)")
ax.set_title("GRB 110721A — light curve with Bayesian blocks (red edges)")
ax.legend(); plt.tight_layout(); plt.show()

## §2 · Fit one block with Band and with Band+Blackbody

**Band** (Band et al. 1993) is the standard empirical GRB continuum: a smoothly-joined
broken power law with low-energy index `α`, high-energy index `β`, and a νFν peak
energy `Ep` (`xp` in astromodels). **Band+Blackbody** adds a thermal (photospheric)
component of temperature `kT` — the extra component that Axelsson (2012) and
Iyyani (2013) reported for *this* burst in time-resolved analyses.

We fit the peak block with both, using **maximum likelihood** (3ML `JointLikelihood`,
Poisson source + Gaussian background = "pgstat"). The identical procedure applies to any
other block or to the full emission window — only the interval changes.

> **Scope caveat.** For clarity this notebook uses just two detectors (n9 + b1) and one
> fixed response, so the exact numbers are a *simplified demonstration* and will differ
> from the full multi-detector pipeline. The **methods and their interpretation** — which
> is the point — carry over unchanged.

> **Likelihood-scale note (important).** 3ML's `jl.current_minimum` is the
> **minus-log-likelihood** `−lnL`, *not* `−2lnL`. So the deviance is
> `−2lnL = 2 × current_minimum`. Every AIC/BIC/DIC below uses that factor of two;
> getting it wrong rescales all the ΔIC gaps.

In [ ]:
def build_plugins(interval):
    plugins = []
    for det, rng in ((NAI, NAI_RANGES), (BGO, BGO_RANGE)):
        b = TimeSeriesBuilder.from_gbm_tte(det, tte_file=tte(det), rsp_file=rsp(det))
        b.set_background_interval(f"{BKG_PRE[0]}-{BKG_PRE[1]}", f"{BKG_POST[0]}-{BKG_POST[1]}")
        b.set_active_time_interval(f"{interval[0]}-{interval[1]}")
        pl = b.to_spectrumlike(); pl.set_active_measurements(*rng)
        plugins.append(pl)
    return plugins

def new_band():
    m = Band(); m.piv = 100.0
    m.alpha.bounds = (-2.0, 2.0); m.beta.bounds = (-5.0, -1.6)
    m.xp.bounds = (10.0, 1e5);    m.xp = 300.0
    return m

def fit_mle(shape, plugins):
    model = Model(PointSource("GRB", 0.0, 0.0, spectral_shape=shape))
    jl = JointLikelihood(model, DataList(*plugins)); jl.set_minimizer("minuit")
    jl.fit(quiet=True)
    return model, jl

# --- fit the PEAK BLOCK (this is "fitting its Bayesian block") ---
plugins = build_plugins(PK)
ndata = int(sum(np.sum(p.mask) for p in plugins))     # active spectral channels

band = new_band()
mB, jlB = fit_mle(band, plugins)

band2 = new_band()
bb = Blackbody(); bb.kT = 30.0; bb.kT.bounds = (1.0, 200.0)
mBB, jlBB = fit_mle(band2 + bb, plugins)

def m2logL(jl):  return 2.0 * jl.current_minimum          # -2 lnL  (see scale note)
def kfree(m):    return len(m.free_parameters)

print(f"active channels n = {ndata}")
print(f"Band     : -2lnL = {m2logL(jlB):8.2f}   k = {kfree(mB)}")
print(f"Band+BB  : -2lnL = {m2logL(jlBB):8.2f}   k = {kfree(mBB)}")
print()
print("Band+BB parameters (MLE):")
print(jlBB.results.get_data_frame()[["value", "negative_error", "positive_error"]])

## §3 · AIC, BIC, DIC — what they are and what they assume

All three trade **goodness of fit** (via the likelihood) against **complexity** (number
of parameters `k`). Lower is better. They differ in *how* they penalise complexity and
in what they assume.

Let `L̂` be the maximum likelihood, `k` the number of free parameters, `n` the number of
data points, and `D(θ) = −2 lnL(θ)` the *deviance*.

| criterion | formula | penalty per parameter | assumes | answers |
|---|---|---|---|---|
| **AIC** | `−2 lnL̂ + 2k` | `2` | large `n`, nested/regular models | which model **predicts** best (KL divergence) |
| **AICc** | `AIC + 2k(k+1)/(n−k−1)` | grows as `n→k` | small-sample correction to AIC | same, safe when `n` not ≫ `k` |
| **BIC** | `−2 lnL̂ + k·ln n` | `ln n` (≈`5.5` here) | one *true* model in the set, flat-ish prior | which model is most **probable** (Bayes factor approx) |
| **DIC** | `D(θ̄) + 2·p_D` | effective, `p_D` | posterior ≈ well-behaved | Bayesian fit-vs-complexity, uses the whole posterior |

Key intuitions:
- **AIC vs BIC**: `ln n` > `2` whenever `n > 7`, so **BIC punishes extra parameters harder**
  than AIC. A component AIC "likes" can be one BIC rejects — that gap *is* the answer to
  "is the blackbody worth it?", not a contradiction.
- **`Δ` interpretation** (Burnham & Anderson): `ΔAIC` of `0–2` ≈ indistinguishable,
  `4–7` = considerably less support, `>10` = essentially none.
- **DIC** needs the **posterior**. `p_D = D̄ − D(θ̄)` = (mean deviance) − (deviance at the
  posterior mean) = the *effective number of parameters*; it can be non-integer, and it
  measures how much the data actually constrained the model.

In [ ]:
def aic(m2, k):        return m2 + 2*k
def aicc(m2, k, n):    return m2 + 2*k + (2*k*(k+1))/max(n-k-1, 1)
def bic(m2, k, n):     return m2 + k*np.log(n)

rows = []
for name, m, jl in [("Band", mB, jlB), ("Band+BB", mBB, jlBB)]:
    m2, k = m2logL(jl), kfree(m)
    rows.append((name, m2, k, aic(m2, k), aicc(m2, k, ndata), bic(m2, k, ndata)))

print(f"{'model':9s} {'-2lnL':>9s} {'k':>2s} {'AIC':>9s} {'AICc':>9s} {'BIC':>9s}")
for r in rows:
    print(f"{r[0]:9s} {r[1]:9.2f} {r[2]:2d} {r[3]:9.2f} {r[4]:9.2f} {r[5]:9.2f}")

dAIC = rows[1][3] - rows[0][3]      # Band+BB minus Band
dBIC = rows[1][5] - rows[0][5]
print(f"\nΔAIC (Band+BB − Band) = {dAIC:+.2f}")
print(f"ΔBIC (Band+BB − Band) = {dBIC:+.2f}")
print("negative ΔIC ⇒ Band+BB preferred by that criterion; the ln(n) vs 2 penalty gap "
      f"= {np.log(ndata):.2f} vs 2 per extra parameter.")

### DIC from the posterior

We now run a short **MCMC** (emcee) for each model to get the posterior, then compute
`DIC = D(θ̄) + 2 p_D` with `p_D = D̄ − D(θ̄)`. The deviance is evaluated directly from the
plugins' log-likelihood (`−2 Σ get_log_like`), which we first **calibrate** against the
MLE so the scale is guaranteed correct. (Sampler settings are small so it runs quickly;
increase `n_iterations` for smoother posteriors.)

In [ ]:
def set_priors_band(m):
    m.K.prior     = Log_uniform_prior(lower_bound=1e-4, upper_bound=1e2)
    m.alpha.prior = Uniform_prior(lower_bound=-2, upper_bound=2)
    m.xp.prior    = Log_uniform_prior(lower_bound=10, upper_bound=1e5)
    m.beta.prior  = Uniform_prior(lower_bound=-5, upper_bound=-1.6)

def deviance(model, plugins):
    return -2.0 * sum(p.get_log_like() for p in plugins)

def run_dic(shape, plugins, label):
    model = Model(PointSource("GRB", 0.0, 0.0, spectral_shape=shape))
    # calibrate the deviance scale against a quick MLE
    jl = JointLikelihood(model, DataList(*plugins)); jl.set_minimizer("minuit"); jl.fit(quiet=True)
    assert np.isclose(deviance(model, plugins), 2*jl.current_minimum, atol=1e-2), "scale mismatch"
    ba = BayesianAnalysis(model, DataList(*plugins))
    ba.set_sampler("emcee"); ba.sampler.setup(n_walkers=20, n_burn_in=100, n_iterations=300)
    ba.sample(quiet=True)
    samp = ba.samples                                   # dict: path -> array (natural units)
    keys = list(samp.keys()); N = len(next(iter(samp.values())))
    idx = np.random.choice(N, size=min(300, N), replace=False)
    Ds = np.empty(len(idx))
    for j, i in enumerate(idx):
        for kname in keys: model[kname].value = samp[kname][i]
        Ds[j] = deviance(model, plugins)
    for kname in keys: model[kname].value = np.mean(samp[kname])   # posterior mean θ̄
    Dhat = deviance(model, plugins)
    Dbar = Ds.mean(); pD = Dbar - Dhat
    print(f"{label:9s}: D(θ̄)={Dhat:8.2f}  D̄={Dbar:8.2f}  p_D={pD:5.2f}  DIC={Dhat+2*pD:8.2f}")
    return Dhat + 2*pD, pD, samp

band_d  = new_band(); set_priors_band(band_d)
DIC_B, pD_B, _ = run_dic(band_d, build_plugins(PK), "Band")

band_d2 = new_band(); set_priors_band(band_d2)
bb_d = Blackbody(); bb_d.kT = 30.0; bb_d.kT.bounds = (1, 200)
bb_d.K.prior  = Log_uniform_prior(lower_bound=1e-8, upper_bound=1e2)
bb_d.kT.prior = Log_uniform_prior(lower_bound=1, upper_bound=200)
DIC_BB, pD_BB, bb_samples = run_dic(band_d2 + bb_d, build_plugins(PK), "Band+BB")

print(f"\nΔDIC (Band+BB − Band) = {DIC_BB - DIC_B:+.2f}   "
      f"(p_D: Band={pD_B:.2f}, Band+BB={pD_BB:.2f} effective params)")
print("Compare the sign/size of ΔAIC, ΔBIC, ΔDIC: do the three criteria agree on the BB?")

## §4 · Error propagation and why covariance matters

A fit gives parameters `θ` with a **covariance matrix** `Σ` (diagonal = variances,
off-diagonal = covariances). For any derived quantity `g(θ)` the **delta method**
(first-order Taylor) gives

$$\sigma_g^2 \;=\; \nabla g^{\top}\,\Sigma\,\nabla g
\;=\;\sum_i\Big(\frac{\partial g}{\partial\theta_i}\Big)^2\sigma_i^2
\;+\;2\sum_{i<j}\frac{\partial g}{\partial\theta_i}\frac{\partial g}{\partial\theta_j}\,\mathrm{Cov}(\theta_i,\theta_j).$$

Two consequences we will *show* numerically:

1. **The dex rule.** For `g = log10(v)`, `∂g/∂v = 1/(v ln10)`, so a symmetric linear
   error `σ_v` becomes `σ_v /(v·ln10)` in dex. (This is exactly the `derr_dex` used in
   the pipeline's correlation plots.)
2. **Ignoring the off-diagonal term is wrong when parameters are correlated.** Band's
   `Ep` and the blackbody `kT` come from the *same fit* and are correlated. For any
   quantity combining them (e.g. the ratio `kT/Ep`), dropping `Cov(Ep,kT)` gives the
   wrong error bar. We verify the delta method against a **Monte-Carlo** draw from the
   full multivariate normal `N(θ̂, Σ)`.

In [ ]:
# ---- use the Band+BB POSTERIOR samples (natural units) for a correct covariance ----
# NOTE: 3ML's results.covariance_matrix is in the fitter's INTERNAL (transformed) space
# for bounded parameters, so it must NOT be read as the natural-unit covariance. The
# posterior samples ARE in natural units, so we take Ep and kT straight from §3's MCMC.
kEp = [k for k in bb_samples if k.endswith("xp_1")][0]
kkT = [k for k in bb_samples if k.endswith("kT_2")][0]
Ep_s, kT_s = np.asarray(bb_samples[kEp]), np.asarray(bb_samples[kkT])
Ep, kT = Ep_s.mean(), kT_s.mean()
C = np.cov(np.vstack([Ep_s, kT_s]))                 # 2x2 covariance, natural units
sEp, skT, cEpkT = np.sqrt(C[0,0]), np.sqrt(C[1,1]), C[0,1]
rho_fit = cEpkT / (sEp * skT)
print(f"Ep = {Ep:7.1f} ± {sEp:5.1f} keV   kT = {kT:6.2f} ± {skT:4.2f} keV")
print(f"Cov(Ep,kT) = {cEpkT:+.2f}  ⇒  corr(Ep,kT) = {rho_fit:+.2f}   (from the Band+BB posterior)")

# (1) the dex rule: err/(v·ln10) vs the posterior spread of log10(Ep)
dex_rule = sEp / (Ep * np.log(10))
dex_post = np.std(np.log10(Ep_s[Ep_s > 0]))
print(f"\nσ(log10 Ep):  err/(Ep·ln10) = {dex_rule:.4f} dex   posterior std = {dex_post:.4f} dex")

# (2) error on the ratio g = kT/Ep, WITH vs WITHOUT the covariance term
g = kT / Ep
dg_dEp, dg_dkT = -kT/Ep**2, 1.0/Ep
var_nocov = dg_dEp**2 * sEp**2 + dg_dkT**2 * skT**2
var_cov   = var_nocov + 2*dg_dEp*dg_dkT*cEpkT
g_post = np.std(kT_s / Ep_s)                          # the posterior "truth"
print(f"\nratio kT/Ep = {g:.4f}")
print(f"  σ_g  ignoring covariance = {np.sqrt(var_nocov):.4e}")
print(f"  σ_g  WITH   covariance   = {np.sqrt(var_cov):.4e}   (delta method)")
print(f"  σ_g  posterior truth     = {g_post:.4e}")
print(f"  → covariance changes σ_g by {100*(np.sqrt(var_cov)/np.sqrt(var_nocov)-1):+.1f}%; "
      "the covariance-aware value matches the posterior. Ignoring Cov mis-states the error.")

fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.6))
ax[0].scatter(Ep_s, kT_s, s=3, alpha=0.15, color="#34699a")
ax[0].set_xlabel("Ep (keV)"); ax[0].set_ylabel("kT (keV)")
ax[0].set_title(f"joint posterior  corr={rho_fit:+.2f}")
ax[1].hist(kT_s/Ep_s, bins=100, color="#b3202c"); ax[1].axvline(g, color="k", lw=1)
ax[1].set_title("kT/Ep posterior (uses full Σ)"); ax[1].set_xlabel("kT/Ep")
plt.tight_layout(); plt.show()

## §5 · Correlation, the Spearman coefficient, and its p-value

**Pearson `r`** measures *linear* association. **Spearman `ρ`** is exactly Pearson's `r`
computed on the **ranks** of the data — so it measures any *monotonic* association and is
invariant under any monotonic transform (log, etc.) and robust to outliers. That
rank-invariance is why taking `log10` before correlating (as the pipeline does) leaves
`ρ` unchanged; the log matters only for the *plot* and for a fitted *slope*.

### Where the p-value comes from
`scipy.stats.spearmanr(x, y)` returns `(ρ, p)`. With default arguments it does **not**
run a permutation test — it uses the **Student-t approximation** to the null distribution:

$$t = \rho\sqrt{\frac{n-2}{1-\rho^2}},\qquad p = 2\,\big[1-F_{t,\,n-2}(|t|)\big].$$

`p` is the probability, **under H₀: ρ_true = 0** (no monotonic association), of seeing a
`|ρ|` at least this large by chance from `n` **independent** pairs. We verify the three
routes agree: scipy `p` == the t-formula == a brute-force permutation p-value.

### The assumptions — and how they fail in practice
1. **Independent pairs.** Time-resolved spectral bins from the *same* burst are **not**
   independent; pooling them inflates `n` and makes `p` far too small (anti-conservative).
2. **Error-free points.** `ρ`/`p` treat each `(x,y)` as exact. Real measurement error
   **attenuates** `ρ` toward 0 (regression dilution) — so a nominal `ρ` *understates* a
   real correlation *if* errors are independent…
3. **…but correlated errors can *manufacture* a correlation.** If the two quantities
   share fit covariance (as Ep and kT do — see §4), the errors are correlated and can
   create a **spurious** `ρ` from data with no intrinsic relation. `p` cannot see this.

In [ ]:
rng = np.random.default_rng(1)

# --- Spearman = Pearson on ranks; monotonic (log) invariance ---
x = rng.normal(size=60); y = x**3 + rng.normal(scale=2, size=60)     # monotonic, nonlinear
rho_manual = np.corrcoef(stats.rankdata(x), stats.rankdata(y))[0, 1]
rho_sc, p_sc = stats.spearmanr(x, y)
pear, _ = stats.pearsonr(x, y)
print(f"Pearson r = {pear:.3f}   Spearman ρ = {rho_sc:.3f}   ρ via rank-Pearson = {rho_manual:.3f}")
xp = np.exp(x)                                                       # positive → log it
print(f"ρ invariant under log10:  {np.isclose(stats.spearmanr(xp, np.exp(y))[0], rho_sc)}")

# --- the p-value three ways ---
n = len(x); t = rho_sc*np.sqrt((n-2)/(1-rho_sc**2)); p_t = 2*stats.t.sf(abs(t), n-2)
B = 20000; cnt = sum(abs(stats.spearmanr(x, rng.permutation(y))[0]) >= abs(rho_sc) for _ in range(B))
p_perm = (cnt + 1) / (B + 1)
print(f"p:  scipy = {p_sc:.2e}   t-approx = {p_t:.2e}   permutation = {p_perm:.2e}")

# --- (2) attenuation from independent measurement error ---
N = 400
kt = rng.uniform(10, 60, N); ep = 3.0*kt + rng.normal(scale=25, size=N)   # true positive relation
rho_true = stats.spearmanr(kt, ep)[0]
kt_o = kt + rng.normal(scale=18, size=N); ep_o = ep + rng.normal(scale=70, size=N)  # add errors
rho_att = stats.spearmanr(kt_o, ep_o)[0]

# --- (3) spurious correlation from covariant errors on a NULL relation ---
# Truth: kt0 and ep0 are INDEPENDENT (no relation). We add measurement errors whose
# Ep/kT components are correlated (as a same-fit Ep–kT covariance makes them), and watch
# a fake correlation appear. Shown at the fit's own corr and at a strong illustrative corr.
kt0 = rng.uniform(10, 60, N); ep0 = rng.uniform(100, 400, N)
def spurious(rr):
    L = np.linalg.cholesky([[1, rr], [rr, 1]])
    e = L @ rng.normal(size=(2, N))
    return kt0 + 6*e[0], ep0 + 60*e[1]
print(f"\ntrue ρ (kt,ep)             = {rho_true:+.3f}")
print(f"ρ after independent errors  = {rho_att:+.3f}   ← attenuated toward 0")
for rr in (rho_fit, 0.8):
    ks, es = spurious(rr)
    print(f"ρ on NULL data, errors corr={rr:+.2f} = {stats.spearmanr(ks, es)[0]:+.3f}   ← SPURIOUS")
kt_s, ep_s = spurious(0.8)                                                # strong case for the plot

fig, ax = plt.subplots(1, 3, figsize=(11, 3.3))
ax[0].scatter(kt, ep, s=8, color="#2a7a2a"); ax[0].set_title(f"true relation  ρ={rho_true:+.2f}")
ax[1].scatter(kt_o, ep_o, s=8, color="#b38f00"); ax[1].set_title(f"+ indep. errors  ρ={rho_att:+.2f}")
ax[2].scatter(kt_s, ep_s, s=8, color="#b3202c")
ax[2].set_title(f"NULL + corr. errors (r=0.8)  ρ={stats.spearmanr(kt_s, ep_s)[0]:+.2f}")
for a in ax: a.set_xlabel("kT (proxy)"); a.set_ylabel("Ep (proxy)")
plt.tight_layout(); plt.show()

## §6 · Synthesis — what you may and may not claim

- **Model choice (§2–3).** Quote **all** of `ΔAIC`, `ΔBIC`, `ΔDIC`, plus `p_D`, not one
  number. When they disagree, that disagreement *is* the finding: AIC rewards predictive
  fit, BIC is stricter on parameters, DIC uses the whole posterior. For a marginal
  blackbody, expect AIC to be keener than BIC — say so explicitly.

- **Errors (§4).** Always propagate with the **covariance matrix**, not just the diagonal.
  Ep and kT here are correlated at `ρ_fit ≈ +0.7` (see §4); any derived quantity or Ep–kT
  comparison that ignores that off-diagonal term has the wrong uncertainty.

- **Correlation (§5).** A Spearman `ρ` with a tiny `p` from pooled, error-free points is
  **not** evidence of a physical Ep–kT relation:
  - the `p`-value assumes **independent, exact** pairs — pooled time-bins violate both;
  - independent errors **hide** a real correlation (attenuation);
  - the **same-fit Ep–kT covariance can manufacture** a correlation from nothing.
  The defensible statistics are: a **per-burst** (not pooled) test, a **block/bootstrap**
  that resamples whole bursts, and — for a slope — an **errors-in-variables** fit
  (ODR / D'Agostini with intrinsic scatter) that folds in `σ_Ep`, `σ_kT`, and their
  covariance. Report `ρ` as *descriptive*, and let the covariance-aware analysis carry
  any physical claim.

*(All numbers above are computed live from GRB 110721A + controlled synthetic demos; the
point is the method, which transfers to any burst and any parameter pair.)*